# Photos Library Metadata-Based Ultimate Thorough Merge — Repair Plan v0.1

Purpose: compare a crash-before **backup** Photos Library and the live **current** Photos Library at the **content identity** level, then generate metadata repair plans.

This notebook is deliberately **dry-run only**:

- It does **not** delete duplicate assets.
- It does **not** modify either Photos Library.
- It does **not** require `photo_library_asset_unique_id` to be unique inside a library.
- Same-identity duplicate assets are treated as one content group for comparison.
- Metadata is unioned per content group for comparison/reporting.

Main outputs:

- `data/ultimate_thorough_merge_reports/internal_duplicate_groups.csv`
- `data/ultimate_thorough_merge_reports/key_collision_groups.csv`
- `data/ultimate_thorough_merge_reports/repair_plan_current_missing_metadata.csv`
- `data/ultimate_thorough_merge_reports/content_identity_comparison_summary.csv`


In [ ]:
# ============================================================
# Section 1: Settings
# ============================================================

from pathlib import Path
from datetime import datetime
from collections import defaultdict, Counter
import csv
import gzip
import hashlib
import json
import os
import pickle
import time

import osxphotos

from explorephotoslibrary import *

USE_INVENTORY_CACHE = True
FORCE_REBUILD_INVENTORY_KEYS = set()
FORCE_RESELECT_LIBRARY_PATHS = False

# Generic role-based cache keys. These are NOT tied to any specific date.
BACKUP_LIBRARY_KEY = "source_backup"
CURRENT_LIBRARY_KEY = "target_current"

LIBRARY_PROMPTS = {
    BACKUP_LIBRARY_KEY: "Select SOURCE / BACKUP Photos Library",
    CURRENT_LIBRARY_KEY: "Select TARGET / CURRENT Photos Library",
}

DEFAULT_INITIAL_DIRS = {
    BACKUP_LIBRARY_KEY: "/Volumes",
    CURRENT_LIBRARY_KEY: str(Path.home() / "Pictures"),
}

LOCAL_CONFIG_PATH = Path("data/local_config/ultimate_thorough_merge_library_paths.json")
REPORT_DIR = Path("data/ultimate_thorough_merge_reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Full SHA verification is expensive for huge movie libraries.
# v0.1 defaults:
# - always SHA-check internal duplicate groups
# - optionally SHA-check common backup/current identities
SHA_CHECK_INTERNAL_DUPLICATES = True
SHA_CHECK_COMMON_IDENTITIES = False

MAX_ROWS_TO_PRINT = 30


In [ ]:
# ============================================================
# Section 2: Small local helpers
# ============================================================

def load_json_file(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def save_json_file(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2, sort_keys=True)


def choose_path_with_tkinter(initial_dir, prompt):
    try:
        import tkinter as tk
        from tkinter import filedialog

        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        selected_path = filedialog.askdirectory(
            initialdir=str(initial_dir),
            title=prompt,
        )
        root.destroy()
        if selected_path:
            return selected_path
    except Exception as exc:
        print("Tkinter picker failed:", repr(exc))

    print(prompt)
    print("Paste Photos Library path manually:")
    return input("> ").strip()


def get_library_path(library_key):
    history = load_json_file(LOCAL_CONFIG_PATH, default={}) or {}
    saved_path = history.get(library_key)

    if saved_path and Path(saved_path).exists() and not FORCE_RESELECT_LIBRARY_PATHS:
        print("=" * 80)
        print(f"Use saved Photos Library path for: {library_key}")
        print("=" * 80)
        print(saved_path)
        print()
        return Path(saved_path)

    if saved_path:
        initial_dir = Path(saved_path).parent
    else:
        initial_dir = Path(DEFAULT_INITIAL_DIRS.get(library_key, "/Volumes"))

    prompt = LIBRARY_PROMPTS.get(library_key, f"Select Photos Library for: {library_key}")
    selected_path = choose_path_with_tkinter(initial_dir=initial_dir, prompt=prompt)

    if not selected_path:
        raise RuntimeError(f"No Photos Library path selected for {library_key}")

    selected_path = Path(selected_path)
    if not selected_path.exists():
        raise FileNotFoundError(selected_path)

    history[library_key] = str(selected_path)
    history[f"{library_key}_selected_at"] = datetime.now().isoformat()
    save_json_file(LOCAL_CONFIG_PATH, history)

    print(f"{library_key} library path:", selected_path)
    print()
    return selected_path


def load_or_build_inventory(library_key):
    library_path = get_library_path(library_key)
    should_rebuild = library_key in FORCE_REBUILD_INVENTORY_KEYS

    if USE_INVENTORY_CACHE and not should_rebuild:
        print("=" * 80)
        print(f"Load inventory cache: {library_key}")
        print("=" * 80)
        try:
            inventory = load_inventory_cache(library_key)
            return inventory
        except FileNotFoundError:
            print(f"Cache not found for {library_key}. Build inventory instead.")
            print()

    print("=" * 80)
    print(f"Build inventory: {library_key}")
    print("=" * 80)
    osx_assets = osxphotos.PhotosDB(str(library_path)).photos()
    print(f"{library_key} osx asset count:", len(osx_assets))

    inventory = build_inventory(osx_assets)
    fill_photo_library_asset_unique_ids(inventory)

    print()
    print(f"{library_key} inventory summary")
    print("-" * 80)
    print_inventory_summary(inventory)
    save_inventory_cache(inventory, library_key)
    return inventory


In [ ]:
# ============================================================
# Section 3: Load or build inventories
# ============================================================

inventory_backup = load_or_build_inventory(BACKUP_LIBRARY_KEY)
print()
inventory_current = load_or_build_inventory(CURRENT_LIBRARY_KEY)

# Make sure identity fields are freshly filled even when loaded from cache.
fill_photo_library_asset_unique_ids(inventory_backup)
fill_photo_library_asset_unique_ids(inventory_current)


In [ ]:
# ============================================================
# Section 4: Metadata extraction helpers
# ============================================================

def compute_sha256_for_asset(asset, chunk_size=1024 * 1024):
    if asset.get("content_sha256"):
        return asset["content_sha256"]

    path = asset.get("path")
    if not path or not os.path.exists(path):
        return None

    sha256 = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            sha256.update(chunk)

    digest = sha256.hexdigest()
    asset["content_sha256"] = digest
    return digest


def asset_album_titles(asset):
    titles = []
    for album in (asset.get("albums") or {}).values():
        title = album.get("title")
        if title:
            titles.append(title)
    return sorted(set(titles))


def asset_folder_paths(asset):
    paths = []
    for folder in (asset.get("folders") or {}).values():
        path = folder.get("path") or folder.get("title")
        if path:
            paths.append(path)
    return sorted(set(paths))


def asset_keywords(asset):
    return sorted(set(asset.get("keywords") or []))


def choose_representative_asset(assets):
    # Prefer richer metadata, then earlier date_added.
    def score(asset):
        metadata_score = (
            len(asset_album_titles(asset)) * 100
            + len(asset_folder_paths(asset)) * 100
            + len(asset_keywords(asset)) * 10
            + (1 if asset.get("description") else 0)
            + (1 if asset.get("favorite") else 0)
            + (1 if asset.get("hidden") else 0)
        )
        date_added = asset.get("date_added") or "9999-99-99"
        return (-metadata_score, date_added, asset.get("uuid") or "")

    return sorted(assets, key=score)[0]


def union_metadata_for_assets(assets):
    album_titles = set()
    folder_paths = set()
    keywords = set()
    descriptions = set()
    titles = set()

    for asset in assets:
        album_titles.update(asset_album_titles(asset))
        folder_paths.update(asset_folder_paths(asset))
        keywords.update(asset_keywords(asset))
        if asset.get("description"):
            descriptions.add(asset.get("description"))
        if asset.get("title"):
            titles.add(asset.get("title"))

    return {
        "album_titles": sorted(album_titles),
        "folder_paths": sorted(folder_paths),
        "keywords": sorted(keywords),
        "descriptions": sorted(descriptions),
        "titles": sorted(titles),
        "favorite_any": any(bool(asset.get("favorite")) for asset in assets),
        "hidden_any": any(bool(asset.get("hidden")) for asset in assets),
    }


def join_values(values):
    if values is None:
        return ""
    if isinstance(values, (list, tuple, set)):
        return " | ".join(str(v) for v in values)
    return str(values)


In [ ]:
# ============================================================
# Section 5: Build content identity groups
# ============================================================

def build_identity_groups(inventory, library_label):
    unique_id_to_assets = defaultdict(list)
    assets_without_unique_id = []

    for asset in inventory["assets"]:
        unique_id = asset.get("photo_library_asset_unique_id")
        if unique_id is None:
            assets_without_unique_id.append(asset)
            continue
        unique_id_to_assets[unique_id].append(asset)

    identity_groups = {}
    internal_duplicate_rows = []
    key_collision_rows = []

    for unique_id, assets in unique_id_to_assets.items():
        sha_to_assets = defaultdict(list)

        if SHA_CHECK_INTERNAL_DUPLICATES and len(assets) > 1:
            for asset in assets:
                sha256 = compute_sha256_for_asset(asset)
                sha_to_assets[sha256].append(asset)
        else:
            sha_to_assets[None] = assets

        # If a duplicated unique_id splits into multiple SHA values, this is a real key collision.
        non_null_sha_values = [sha for sha in sha_to_assets.keys() if sha is not None]
        if len(non_null_sha_values) > 1:
            for sha256, sha_assets in sha_to_assets.items():
                for asset in sha_assets:
                    key_collision_rows.append({
                        "library_label": library_label,
                        "unique_id": repr(unique_id),
                        "sha256": sha256,
                        "uuid": asset.get("uuid"),
                        "original_filename": asset.get("original_filename"),
                        "date": asset.get("date"),
                        "date_added": asset.get("date_added"),
                        "path": asset.get("path"),
                    })
            # Still keep groups, but mark comparison unsafe for this identity.

        # Most groups will be one SHA bucket, or one unchecked bucket.
        # Use one group per unique_id for v0.1; duplicate structure remains in member_assets.
        representative = choose_representative_asset(assets)
        metadata_union = union_metadata_for_assets(assets)

        group = {
            "library_label": library_label,
            "unique_id": unique_id,
            "asset_count": len(assets),
            "representative_uuid": representative.get("uuid"),
            "representative_original_filename": representative.get("original_filename"),
            "representative_date": representative.get("date"),
            "representative_date_added": representative.get("date_added"),
            "representative_path": representative.get("path"),
            "metadata_union": metadata_union,
            "member_assets": assets,
            "sha_values_checked": sorted(str(sha) for sha in sha_to_assets.keys()),
            "has_internal_key_collision": len(non_null_sha_values) > 1,
        }
        identity_groups[unique_id] = group

        if len(assets) > 1:
            internal_duplicate_rows.append({
                "library_label": library_label,
                "unique_id": repr(unique_id),
                "asset_count": len(assets),
                "sha_values_checked": join_values(group["sha_values_checked"]),
                "has_internal_key_collision": group["has_internal_key_collision"],
                "representative_uuid": group["representative_uuid"],
                "representative_original_filename": group["representative_original_filename"],
                "album_titles_union": join_values(metadata_union["album_titles"]),
                "folder_paths_union": join_values(metadata_union["folder_paths"]),
                "keywords_union": join_values(metadata_union["keywords"]),
                "member_uuids": join_values([asset.get("uuid") for asset in assets]),
            })

    return {
        "library_label": library_label,
        "identity_groups": identity_groups,
        "assets_without_unique_id": assets_without_unique_id,
        "internal_duplicate_rows": internal_duplicate_rows,
        "key_collision_rows": key_collision_rows,
    }


backup_identity_index = build_identity_groups(inventory_backup, "backup")
current_identity_index = build_identity_groups(inventory_current, "current")

print("backup identity group count:", len(backup_identity_index["identity_groups"]))
print("backup internal duplicate group count:", len(backup_identity_index["internal_duplicate_rows"]))
print("backup internal key collision rows:", len(backup_identity_index["key_collision_rows"]))
print()
print("current identity group count:", len(current_identity_index["identity_groups"]))
print("current internal duplicate group count:", len(current_identity_index["internal_duplicate_rows"]))
print("current internal key collision rows:", len(current_identity_index["key_collision_rows"]))


In [ ]:
# ============================================================
# Section 6: Cross-library content identity comparison
# ============================================================

def compare_identity_sets(backup_index, current_index):
    backup_groups = backup_index["identity_groups"]
    current_groups = current_index["identity_groups"]

    backup_keys = set(backup_groups.keys())
    current_keys = set(current_groups.keys())

    common_keys = sorted(backup_keys & current_keys, key=repr)
    backup_only_keys = sorted(backup_keys - current_keys, key=repr)
    current_only_keys = sorted(current_keys - backup_keys, key=repr)

    summary_rows = []
    repair_rows = []

    for unique_id in common_keys:
        backup_group = backup_groups[unique_id]
        current_group = current_groups[unique_id]

        bmeta = backup_group["metadata_union"]
        cmeta = current_group["metadata_union"]

        for field_name in ["album_titles", "folder_paths", "keywords"]:
            backup_values = set(bmeta[field_name])
            current_values = set(cmeta[field_name])

            missing_in_current = sorted(backup_values - current_values)
            new_in_current = sorted(current_values - backup_values)

            for value in missing_in_current:
                repair_rows.append({
                    "repair_type": f"CURRENT_MISSING_{field_name}",
                    "unique_id": repr(unique_id),
                    "field_name": field_name,
                    "missing_value": value,
                    "backup_representative_uuid": backup_group["representative_uuid"],
                    "current_representative_uuid": current_group["representative_uuid"],
                    "backup_asset_count_for_identity": backup_group["asset_count"],
                    "current_asset_count_for_identity": current_group["asset_count"],
                    "backup_original_filename": backup_group["representative_original_filename"],
                    "current_original_filename": current_group["representative_original_filename"],
                    "backup_date": backup_group["representative_date"],
                    "current_date": current_group["representative_date"],
                    "notes": "Dry-run repair plan only. Does not modify Photos Library.",
                })

            if missing_in_current or new_in_current:
                summary_rows.append({
                    "change_type": f"METADATA_UNION_DIFF_{field_name}",
                    "unique_id": repr(unique_id),
                    "field_name": field_name,
                    "backup_values": join_values(sorted(backup_values)),
                    "current_values": join_values(sorted(current_values)),
                    "missing_in_current": join_values(missing_in_current),
                    "new_in_current": join_values(new_in_current),
                    "backup_asset_count_for_identity": backup_group["asset_count"],
                    "current_asset_count_for_identity": current_group["asset_count"],
                    "backup_representative_uuid": backup_group["representative_uuid"],
                    "current_representative_uuid": current_group["representative_uuid"],
                    "original_filename": backup_group["representative_original_filename"],
                    "date": backup_group["representative_date"],
                })

        # Scalar-ish metadata, conservative v0.1.
        if bmeta["favorite_any"] and not cmeta["favorite_any"]:
            repair_rows.append({
                "repair_type": "CURRENT_MISSING_favorite_true",
                "unique_id": repr(unique_id),
                "field_name": "favorite",
                "missing_value": "True",
                "backup_representative_uuid": backup_group["representative_uuid"],
                "current_representative_uuid": current_group["representative_uuid"],
                "backup_asset_count_for_identity": backup_group["asset_count"],
                "current_asset_count_for_identity": current_group["asset_count"],
                "backup_original_filename": backup_group["representative_original_filename"],
                "current_original_filename": current_group["representative_original_filename"],
                "backup_date": backup_group["representative_date"],
                "current_date": current_group["representative_date"],
                "notes": "Dry-run repair plan only. Does not modify Photos Library.",
            })

    for unique_id in backup_only_keys:
        group = backup_groups[unique_id]
        summary_rows.append({
            "change_type": "CONTENT_IDENTITY_MISSING_FROM_CURRENT",
            "unique_id": repr(unique_id),
            "field_name": "content_identity",
            "backup_values": "exists",
            "current_values": "missing",
            "missing_in_current": "content_identity",
            "new_in_current": "",
            "backup_asset_count_for_identity": group["asset_count"],
            "current_asset_count_for_identity": 0,
            "backup_representative_uuid": group["representative_uuid"],
            "current_representative_uuid": "",
            "original_filename": group["representative_original_filename"],
            "date": group["representative_date"],
        })

    for unique_id in current_only_keys:
        group = current_groups[unique_id]
        summary_rows.append({
            "change_type": "CONTENT_IDENTITY_NEW_IN_CURRENT",
            "unique_id": repr(unique_id),
            "field_name": "content_identity",
            "backup_values": "missing",
            "current_values": "exists",
            "missing_in_current": "",
            "new_in_current": "content_identity",
            "backup_asset_count_for_identity": 0,
            "current_asset_count_for_identity": group["asset_count"],
            "backup_representative_uuid": "",
            "current_representative_uuid": group["representative_uuid"],
            "original_filename": group["representative_original_filename"],
            "date": group["representative_date"],
        })

    return summary_rows, repair_rows


summary_rows, repair_rows = compare_identity_sets(backup_identity_index, current_identity_index)

print("comparison summary row count:", len(summary_rows))
print("repair plan row count:", len(repair_rows))
print()
print("Top change types:")
for change_type, count in Counter(row["change_type"] for row in summary_rows).most_common(20):
    print(count, change_type)
print()
print("Top repair types:")
for repair_type, count in Counter(row["repair_type"] for row in repair_rows).most_common(20):
    print(count, repair_type)


In [ ]:
# ============================================================
# Section 7: Write reports
# ============================================================

def write_csv(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    if not rows:
        path.write_text("", encoding="utf-8")
        print("wrote empty CSV:", path)
        return path

    fieldnames = list(rows[0].keys())
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)

    print("wrote CSV:", path)
    print("row count:", len(rows))
    return path


all_internal_duplicate_rows = (
    backup_identity_index["internal_duplicate_rows"]
    + current_identity_index["internal_duplicate_rows"]
)
all_key_collision_rows = (
    backup_identity_index["key_collision_rows"]
    + current_identity_index["key_collision_rows"]
)

internal_duplicate_report_path = write_csv(
    REPORT_DIR / "internal_duplicate_groups.csv",
    all_internal_duplicate_rows,
)

key_collision_report_path = write_csv(
    REPORT_DIR / "key_collision_groups.csv",
    all_key_collision_rows,
)

summary_report_path = write_csv(
    REPORT_DIR / "content_identity_comparison_summary.csv",
    summary_rows,
)

repair_plan_path = write_csv(
    REPORT_DIR / "repair_plan_current_missing_metadata.csv",
    repair_rows,
)


In [ ]:
# ============================================================
# Section 8: Inspect repair plan quickly
# ============================================================

print("Repair plan preview")
print("=" * 80)

for row in repair_rows[:MAX_ROWS_TO_PRINT]:
    print("repair_type:", row["repair_type"])
    print("field_name:", row["field_name"])
    print("missing_value:", row["missing_value"])
    print("original_filename:", row["backup_original_filename"])
    print("backup uuid:", row["backup_representative_uuid"])
    print("current uuid:", row["current_representative_uuid"])
    print("backup/current asset counts:", row["backup_asset_count_for_identity"], "/", row["current_asset_count_for_identity"])
    print("-" * 80)

if len(repair_rows) > MAX_ROWS_TO_PRINT:
    print("... more rows not printed")


## Next phase, not implemented in v0.1

This notebook only creates a repair plan. The next notebook/module should be a guarded writer:

1. Read `repair_plan_current_missing_metadata.csv`.
2. Validate that each target `current_representative_uuid` still exists.
3. Create missing folder/album structure if needed.
4. Add the current asset to missing albums / folder paths.
5. Apply keywords/favorite/description only when the rule is unambiguous.
6. Write an operation log after every successful change.
7. Support resume after interruption.

This separation is intentional: report generation is safe; modifying a live iCloud Photos Library must be a separate, auditable step.
